In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

ROOT = r"C:\Users\reinh\Downloads\Projects\Rossmann"
RAW_TRAIN = f"{ROOT}/Rossmann/train.csv"
RAW_STORE = f"{ROOT}/Rossmann/store.csv"
PROC = f"{ROOT}/rsmn"

HORIZON = 28
WINDOW_OFFSETS = [0, 28, 56]
MARGINS = [0.15, 0.25, 0.35]

In [2]:
train = pd.read_csv(RAW_TRAIN, dtype={"StateHoliday": str}, parse_dates=["Date"])
store = pd.read_csv(RAW_STORE)
calendar = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")

# Filter stores with >=95% coverage and no missing gap >= 14 days
def max_gap(dates):
    missing = calendar.difference(pd.DatetimeIndex(dates))
    if len(missing) == 0:
        return 0
    return int((pd.Series(missing).diff().dt.days != 1).cumsum().value_counts().max())

coverage = train.groupby("Store")["Date"].nunique() / len(calendar)
gaps = train.groupby("Store")["Date"].apply(max_gap)
eligible = coverage[(coverage >= 0.95) & (gaps < 14)].index.values

TEST_STORES = sorted(np.random.default_rng(42).choice(eligible, size=5, replace=False).tolist())
print("Eligible stores:", len(eligible), "| Test stores:", TEST_STORES)

# Save clean open-day reference table
clean = train.merge(store, on="Store", how="left")
clean = clean[(clean["Open"] == 1) & (clean["Sales"] > 0)].sort_values(["Store", "Date"])
clean.to_csv(f"{PROC}/rossmann_clean.csv", index=False)

Eligible stores: 935 | Test stores: [98, 480, 486, 735, 865]


In [3]:
# Build full calendar grid so 28+ day lags align to exact calendar days
panel = (
    train.set_index(["Store", "Date"])
    .reindex(pd.MultiIndex.from_product([eligible, calendar], names=["Store", "Date"]))
    .reset_index()
    .merge(store, on="Store", how="left")
    .sort_values(["Store", "Date"])
    .reset_index(drop=True)
)

g = panel.groupby("Store")["Sales"]
for k in [28, 35, 42, 49]:
    panel[f"lag_{k}"] = g.shift(k)

panel["same_dow_mean_4w"] = panel[["lag_28", "lag_35", "lag_42", "lag_49"]].mean(axis=1)
panel["roll_mean_28"] = g.transform(lambda s: s.shift(HORIZON).rolling(HORIZON, min_periods=14).mean())

panel["DayOfWeek"] = panel["Date"].dt.dayofweek + 1
panel["month"] = panel["Date"].dt.month
panel["day_of_month"] = panel["Date"].dt.day
panel["week_of_year"] = panel["Date"].dt.isocalendar().week.astype(int)
panel["StateHoliday"] = panel["StateHoliday"].fillna("0").map({"0": 0, "a": 1, "b": 2, "c": 3})
panel["StoreType"] = panel["StoreType"].astype("category").cat.codes
panel["Assortment"] = panel["Assortment"].astype("category").cat.codes
panel["CompetitorMissing"] = panel["CompetitionDistance"].isna().astype(int)
panel["CompetitionDistance"] = panel["CompetitionDistance"].fillna(100_000)

FEATURES = [
    "DayOfWeek", "Promo", "StateHoliday", "SchoolHoliday",
    "month", "day_of_month", "week_of_year",
    "StoreType", "Assortment", "CompetitionDistance", "CompetitorMissing", "Promo2",
    "lag_28", "lag_35", "lag_42", "lag_49", "same_dow_mean_4w", "roll_mean_28",
]

In [4]:
def mape(actual, pred):
    return np.mean(np.abs(actual - pred) / actual) * 100

def wape(actual, pred):
    return np.sum(np.abs(actual - pred)) / np.sum(actual) * 100

def arima_forecast(y_train):
    y = np.log1p(y_train.interpolate(limit_direction="both"))
    m = SARIMAX(y, order=(1, 0, 1), seasonal_order=(1, 0, 1, 7), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    return np.expm1(m.forecast(HORIZON)).clip(lower=0).values

base_rows = []
for off in WINDOW_OFFSETS:
    cut = calendar[-HORIZON - 1 - off]
    end = cut + pd.Timedelta(days=HORIZON)
    for s in TEST_STORES:
        d = panel[panel["Store"] == s].set_index("Date")
        y_tr = d["Sales"].loc[:cut]
        y_te = d.loc[cut + pd.Timedelta(days=1):end]
        keep = (y_te["Open"].eq(1) & (y_te["Sales"] > 0)).values
        actual = y_te["Sales"].values[keep]

        # Same-weekday 4-week mean
        hist = y_tr.tail(HORIZON)
        pred_naive = hist.groupby(hist.index.dayofweek).mean().reindex(y_te.index.dayofweek).values[keep]
        pred_arima = arima_forecast(y_tr)[keep]

        for name, p in [("Same-weekday mean", pred_naive), ("ARIMA", pred_arima)]:
            base_rows.append({"cutoff": cut.date(), "Store": s, "Model": name, "MAPE_%": mape(actual, p), "WAPE_%": wape(actual, p)})

base_df = pd.DataFrame(base_rows)
print(base_df.groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2))
base_df.to_csv(f"{PROC}/baseline_results.csv", index=False)

                   MAPE_%  WAPE_%
Model                            
ARIMA               29.05   31.96
Same-weekday mean   19.92   19.74


In [5]:
def evaluate_gb(cut, feature_cols):
    is_tr = (panel["Date"] <= cut) & (panel["Open"] == 1) & panel["Sales"].notna()
    model = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, min_samples_leaf=100, early_stopping=False, random_state=42)
    model.fit(panel.loc[is_tr, feature_cols], np.log1p(panel.loc[is_tr, "Sales"]))

    end = cut + pd.Timedelta(days=HORIZON)
    ho = panel[(panel["Date"] > cut) & (panel["Date"] <= end) & panel["Store"].isin(TEST_STORES)].copy()
    ho["pred_gb"] = np.expm1(model.predict(ho[feature_cols])).clip(min=0)
    ho["cutoff"] = cut.date()
    return ho[(ho["Open"] == 1) & (ho["Sales"] > 0)][["cutoff", "Store", "Date", "Sales", "pred_gb"]]

daily = pd.concat([evaluate_gb(calendar[-HORIZON - 1 - off], FEATURES) for off in WINDOW_OFFSETS], ignore_index=True)
daily.to_csv(f"{PROC}/holdout_daily_predictions.csv", index=False)

gb_acc = pd.DataFrame([
    {"cutoff": c, "Store": s, "Model": "Gradient boosting", "MAPE_%": mape(d["Sales"], d["pred_gb"]), "WAPE_%": wape(d["Sales"], d["pred_gb"])}
    for (c, s), d in daily.groupby(["cutoff", "Store"])
])

all_acc = pd.concat([gb_acc, base_df], ignore_index=True)
print("=== Average Across All 3 Windows ===")
print(all_acc.groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2).sort_values("MAPE_%"))

=== Average Across All 3 Windows ===
                   MAPE_%  WAPE_%
Model                            
Gradient boosting    9.18    9.04
Same-weekday mean   19.92   19.74
ARIMA               29.05   31.96


In [6]:
# 1. Promotion & Holiday Ablation
NO_PROMO = [f for f in FEATURES if f not in ["Promo", "Promo2", "SchoolHoliday", "StateHoliday"]]
ablation_daily = pd.concat([evaluate_gb(calendar[-HORIZON - 1 - off], NO_PROMO) for off in WINDOW_OFFSETS], ignore_index=True)

ablation_acc = pd.DataFrame([
    {"Model": "GB (No Promo/Holiday)", "MAPE_%": mape(d["Sales"], d["pred_gb"]), "WAPE_%": wape(d["Sales"], d["pred_gb"])}
    for _, d in ablation_daily.groupby(["cutoff", "Store"])
])
print("=== Promotion Ablation ===")
print(ablation_acc[["MAPE_%", "WAPE_%"]].mean().round(2))
ablation_acc.to_csv(f"{PROC}/ablation_results.csv", index=False)

# 2. Inventory Cost Analysis
naive_map = {}
for off in WINDOW_OFFSETS:
    cut = calendar[-HORIZON - 1 - off]
    for s in TEST_STORES:
        hist = panel[(panel["Store"] == s) & (panel["Date"] <= cut)].tail(HORIZON)
        by_dow = hist.groupby("DayOfWeek")["Sales"].mean()
        for dow, val in by_dow.items():
            naive_map[(cut.date(), s, dow)] = val

daily["DayOfWeek"] = pd.to_datetime(daily["Date"]).dt.dayofweek + 1
daily["pred_naive"] = [naive_map[(r.cutoff, r.Store, r.DayOfWeek)] for r in daily.itertuples()]

cost_rows = []
for m in MARGINS:
    for name, col in [("Gradient boosting", "pred_gb"), ("Same-weekday mean", "pred_naive")]:
        over = (daily[col] - daily["Sales"]).clip(lower=0)
        under = (daily["Sales"] - daily[col]).clip(lower=0)
        cost_eur = float(((1 - m) * over + m * under).sum())
        cost_rows.append({"margin": m, "Model": name, "cost_eur": round(cost_eur)})

cost_df = pd.DataFrame(cost_rows)
print("\n=== Inventory Cost (€) ===")
print(cost_df.pivot(index="margin", columns="Model", values="cost_eur"))
cost_df.to_csv(f"{PROC}/inventory_cost_results.csv", index=False)

=== Promotion Ablation ===
MAPE_%    10.70
WAPE_%    10.63
dtype: float64

=== Inventory Cost (€) ===
Model   Gradient boosting  Same-weekday mean
margin                                      
0.15                90481             179866
0.25                90725             184414
0.35                90970             188962
